# What is a Time Series?

Time series data is everywhere. Stock prices tick by the second, weather stations log
temperatures every hour, retailers track daily sales, and governments publish quarterly GDP.
Any variable recorded sequentially over time produces a **time series**, and understanding
these sequences requires specialised methods that respect the temporal ordering of
observations.

This notebook — the first in a comprehensive time series analysis course — introduces the
foundational concepts you will need before fitting any model. The material follows
Chapter 1 of *Forecasting: Principles and Practice* (3rd ed.) by Rob J. Hyndman and
George Athanasopoulos (FPP3).

## Setup

Import the libraries and configure paths used throughout this notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = Path("../data")

# ---------------------------------------------------------------------------
# Plotting defaults
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "lines.linewidth": 1.5,
    "figure.dpi": 100,
})

---

## 1. What Can Be Forecast?

Not everything is equally predictable. The **predictability** of an event or quantity
depends on several factors:

1. **How well we understand the contributing factors** — Do we know what drives the
   variable of interest?
2. **How much data is available** — Do we have a long, clean historical record?
3. **Whether the future resembles the past** — Are the underlying patterns stable, or is
   the environment changing in unprecedented ways?
4. **Whether forecasts themselves affect outcomes** — For example, a forecast of high
   electricity demand may trigger load-shedding that *reduces* demand, invalidating the
   original forecast.

### Electricity demand — highly predictable

Electricity demand is one of the most forecastable quantities in practice. We know its
main drivers (temperature, time of day, day of week, holidays), utilities have decades
of high-frequency data, and the patterns are remarkably stable year after year.

### Exchange rates — nearly unpredictable

Currency exchange rates, by contrast, are notoriously difficult to forecast. The
*efficient market hypothesis* suggests that exchange rates already incorporate all
available information, so future movements are essentially random.

> **Key insight:** *"A good forecasting model captures the way things are changing."*
> The goal is not to predict the future perfectly, but to quantify the patterns and
> uncertainty in a useful way.

---

## 2. Types of Time Series Data

Data used in forecasting and analytics broadly falls into three categories:

| Type | Description | Example |
|------|-------------|---------|
| **Time series** | Observations recorded at regular intervals over time | Monthly airline passengers |
| **Cross-sectional** | Observations collected at a single point in time across many entities | A census snapshot of household income by county |
| **Panel (longitudinal)** | Combines both — multiple entities observed over time | GDP of 50 countries measured quarterly for 20 years |

This course focuses on **time series data**.

Time series can be recorded at many frequencies:

| Frequency | Example |
|-----------|--------|
| Annual | Yearly national population |
| Quarterly | GDP reports |
| Monthly | Airline passengers, unemployment rate |
| Weekly | Retail sales |
| Daily | Stock closing prices |
| Hourly | Electricity demand |
| Sub-hourly | High-frequency trading data, sensor readings |

The following cell demonstrates the distinction between these data types with small,
synthetic examples.

In [ ]:
# ── Time series data: one entity observed over time ─────────────────────────
ts_example = pd.DataFrame({
    "date": pd.date_range("2024-01", periods=6, freq="MS"),
    "monthly_sales": [210, 225, 198, 245, 260, 230],
})
print("Time Series Data")
print(ts_example.to_string(index=False))

print("\n")

# ── Cross-sectional data: many entities at a single point in time ───────────
cs_example = pd.DataFrame({
    "city": ["New York", "London", "Tokyo", "Sydney"],
    "avg_temp_C": [12.3, 10.1, 15.6, 22.4],
})
print("Cross-Sectional Data (snapshot on 2024-06-01)")
print(cs_example.to_string(index=False))

print("\n")

# ── Panel data: multiple entities observed over time ────────────────────────
panel_example = pd.DataFrame({
    "country": ["USA", "USA", "USA", "UK", "UK", "UK"],
    "year":    [2021, 2022, 2023, 2021, 2022, 2023],
    "gdp_trillion_usd": [23.3, 25.5, 27.4, 3.1, 3.1, 3.3],
})
print("Panel Data")
print(panel_example.to_string(index=False))

---

## 3. Components of a Time Series

Most time series can be decomposed into a combination of the following components:

| Component | Description |
|-----------|-------------|
| **Trend** | The long-term increase or decrease in the data. A trend can be linear, exponential, or any smooth, slowly changing function. |
| **Seasonality** | A regular, calendar-related repeating pattern with a **fixed and known period** (e.g., every 12 months, every 7 days). |
| **Cyclicity** | Rises and falls that are **not** of a fixed period. Business cycles typically last 2 or more years and their duration varies. |
| **Noise / Remainder** | Random variation that cannot be attributed to trend, seasonality, or cycles. |

> **Seasonality vs. cyclicity:** The critical distinction is that seasonality has a
> **fixed and known period** tied to the calendar (e.g., summer peaks every year),
> while cyclicity has a **variable and often unknown period** (e.g., economic
> recessions every 5–10 years, but never on a precise schedule).

Let's load the classic **airline passengers** dataset and visualise these components.
This dataset records the monthly totals of international airline passengers from
1949 to 1960.

In [ ]:
# Load airline passengers data
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    parse_dates=["Month"],
    index_col="Month",
)
airline.columns = ["Passengers"]
airline.index.freq = "MS"  # month-start frequency

airline.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))

# Plot the raw series
ax.plot(airline.index, airline["Passengers"], color="#2c7bb6", label="Passengers")

# Overlay a 12-month rolling mean to highlight the trend
trend = airline["Passengers"].rolling(window=12, center=True).mean()
ax.plot(airline.index, trend, color="#d7191c", linewidth=2.5, label="12-month trend")

# Annotate trend
ax.annotate(
    "Upward trend",
    xy=(pd.Timestamp("1957-06"), trend.loc["1957-06"]),
    xytext=(pd.Timestamp("1953-01"), 420),
    fontsize=12,
    fontweight="bold",
    color="#d7191c",
    arrowprops=dict(arrowstyle="->", color="#d7191c", lw=1.5),
)

# Annotate seasonality — point to a summer peak
ax.annotate(
    "Seasonal peak\n(every summer)",
    xy=(pd.Timestamp("1955-07"), airline.loc["1955-07", "Passengers"]),
    xytext=(pd.Timestamp("1950-06"), 380),
    fontsize=12,
    fontweight="bold",
    color="#2c7bb6",
    arrowprops=dict(arrowstyle="->", color="#2c7bb6", lw=1.5),
)

ax.set_title("Monthly Airline Passengers (1949 – 1960)")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of passengers")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

The plot above clearly shows two of the four components:

- **Trend:** Passenger numbers increase steadily over the decade (the red line).
- **Seasonality:** There is a repeating annual pattern with peaks every summer (July–August)
  and troughs in winter. The seasonal amplitude *grows* over time, suggesting a
  **multiplicative** relationship between trend and seasonality.

We do not see obvious cyclicity in this relatively short series, and the noise is the
residual wiggle around the trend-seasonal pattern.

---

## 4. Forecasting, Goals, and Planning

Forecasting is often confused with goal-setting and planning. FPP3 draws a clear
distinction:

| Concept | Definition | Example |
|---------|------------|---------|
| **Forecasting** | Predicting the future as accurately as possible, given all available information | "We expect to sell 10,000 units next quarter" |
| **Goals** | What you *want* to happen — aspirational targets | "We want to sell 15,000 units next quarter" |
| **Planning** | The actions you take in response to forecasts and goals | "We will hire 5 more salespeople to close the gap" |

Forecasts serve different purposes depending on the **time horizon**:

- **Short-term forecasts** (days to weeks): scheduling staff, managing inventory,
  routing deliveries.
- **Medium-term forecasts** (months to a year): resource planning, budgeting,
  procurement.
- **Long-term forecasts** (years): strategic planning, capital investment, market
  entry decisions.

The appropriate model and level of detail often differ across horizons.

---

## 5. Qualitative vs. Quantitative Forecasting

Forecasting methods fall into two broad families:

### Qualitative (judgmental) methods

Used when **no historical data** is available or when the situation is entirely novel:

- **Expert judgment** — leveraging domain knowledge and experience.
- **Delphi method** — structured rounds of anonymous expert opinion, iterating toward
  consensus.
- **Scenario planning** — constructing plausible future narratives and assessing their
  implications.

### Quantitative (statistical) methods

Used when two conditions are met:

1. **Numerical data** about the past is available.
2. It is reasonable to assume that **past patterns will continue** into the future.

Quantitative methods range from simple benchmarks (naive forecasts, moving averages) to
sophisticated machine-learning models. **This course focuses on quantitative methods.**

---

## 6. The Five Steps of a Forecasting Task

Hyndman and Athanasopoulos outline a practical workflow for any forecasting project:

### Step 1 — Problem definition

> *"Often the most difficult part of forecasting."*

Before touching any data, clarify: What exactly needs to be forecast? At what frequency?
For whom? How will the forecast be used? Misunderstanding the problem leads to wasted
effort — or worse, a technically correct forecast that answers the wrong question.

### Step 2 — Gathering information

Two kinds of information are needed:

- **Statistical data** — historical observations of the variable(s) of interest.
- **Domain expertise** — knowledge from people who understand the system being forecast
  (e.g., a supply-chain manager who knows about upcoming promotions).

### Step 3 — Preliminary (exploratory) analysis

> *"Always start by graphing the data!"*

Plot the series. Look for:
- Trends and level shifts
- Seasonal patterns
- Outliers and anomalies
- Structural breaks (e.g., policy changes, pandemics)
- Relationships with other variables

### Step 4 — Choosing and fitting models

Select candidate models (e.g., exponential smoothing, ARIMA, regression) and fit them to
the data. Compare models using appropriate accuracy metrics. No single model dominates in
every situation — always compare multiple approaches.

### Step 5 — Using and evaluating the forecast

Deploy the forecast, then **evaluate it when actual data becomes available**. This
feedback loop is essential: it reveals model weaknesses and guides improvements for the
next forecasting cycle.

---

## 7. The Statistical Forecasting Perspective

From a statistical point of view, the future value of a time series, $y_{T+h}$, is a
**random variable**. We do not know its exact value, but we can describe the set of
values it might take and how likely each value is.

### Forecast distribution

The **forecast distribution** is the probability distribution of $y_{T+h}$ conditional on
everything we have observed up to time $T$:

$$
y_{T+h} \mid y_1, y_2, \dots, y_T
$$

### Point forecasts

The most common summary of the forecast distribution is the **point forecast** — typically
the mean of the distribution:

$$
\hat{y}_{T+h|T} = \mathbb{E}[y_{T+h} \mid y_1, \dots, y_T]
$$

Here $\hat{y}_{T+h|T}$ denotes the $h$-step-ahead point forecast made at time $T$.

### Prediction intervals

A point forecast alone is incomplete — it says nothing about uncertainty.
**Prediction intervals** provide a range within which the future value is expected to
fall with a given probability:

- An **80% prediction interval** contains the true value 80% of the time.
- A **95% prediction interval** is wider and contains the true value 95% of the time.

> *"The further ahead we forecast, the more uncertain we are."*

Prediction intervals naturally widen as the forecast horizon $h$ increases.

The cell below illustrates expanding prediction intervals with a simple synthetic
example.

In [ ]:
# Illustrate how prediction intervals widen with forecast horizon
np.random.seed(42)

# Historical data: a series with slight upward trend + noise
T = 60
t_hist = np.arange(T)
y_hist = 100 + 0.5 * t_hist + np.random.normal(0, 4, T)

# Forecast horizon
H = 24
t_fc = np.arange(T, T + H)
y_fc = 100 + 0.5 * t_fc  # point forecast (trend extrapolation)

# Simulated prediction intervals that widen with h
sigma = 4
h = np.arange(1, H + 1)
interval_80 = 1.28 * sigma * np.sqrt(h)
interval_95 = 1.96 * sigma * np.sqrt(h)

fig, ax = plt.subplots(figsize=(13, 5))

# Historical
ax.plot(t_hist, y_hist, color="#2c7bb6", label="Historical data")

# Point forecast
ax.plot(t_fc, y_fc, color="#d7191c", linewidth=2, label="Point forecast $\\hat{y}_{T+h|T}$")

# 80% interval
ax.fill_between(t_fc, y_fc - interval_80, y_fc + interval_80,
                color="#fdae61", alpha=0.5, label="80% prediction interval")

# 95% interval
ax.fill_between(t_fc, y_fc - interval_95, y_fc + interval_95,
                color="#fdae61", alpha=0.25, label="95% prediction interval")

ax.axvline(T - 0.5, color="grey", linestyle="--", linewidth=1, alpha=0.7)
ax.text(T - 1, ax.get_ylim()[1] - 2, "Forecast origin", ha="right", fontsize=10, color="grey")

ax.set_title("Point Forecast with Expanding Prediction Intervals")
ax.set_xlabel("Time")
ax.set_ylabel("$y_t$")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Notice how the shaded bands fan out as the horizon increases. Early forecasts are
relatively precise, but by the end of the horizon the range of plausible outcomes is
much wider. Communicating this uncertainty honestly is one of the hallmarks of good
forecasting practice.

---

## 8. What's Ahead — Course Roadmap

This notebook covered the conceptual foundations. The rest of the course builds on these
ideas systematically:

| Chapter | Topic |
|---------|-------|
| **02** | Time series graphics — visualisation techniques for seasonal plots, lag plots, and autocorrelation |
| **03** | Time series decomposition — isolating trend, seasonality, and remainder |
| **04** | Features and statistics — numerical summaries of time series properties |
| **05** | Simple forecasting methods — naive, seasonal naive, drift, and moving-average benchmarks |
| **06** | Forecast accuracy and evaluation — train/test splits, cross-validation, error metrics |
| **07** | Exponential smoothing (ETS) — the workhorse of business forecasting |
| **08** | ARIMA models — modelling autocorrelation with differencing |
| **09** | Regression with time series — using external predictors and dynamic regression |
| **10** | Advanced topics — state-space models, hierarchical forecasting, and machine-learning approaches |

Each chapter includes hands-on code, real datasets, and exercises. Let's get started!